# Notebook 1b — Complementos da parte 1
Faz o que o plano pedia à pessoa 1 e que não está no notebook 1. Usa o código dela (`src/parte1_base.py`) e as divisões que o notebook 1 guardou (`results/notebook1/splits_seed42.json`), sem alterar nada.

| Secção | O que faz | Saída |
|---|---|---|
| 1 | Verificação dos dados: casos, especialidades, campos vazios, textos repetidos dentro da mesma divisão | – |
| 2 | Os 50 casos que o pandas perde ao ler o CSV, recuperados linha a linha | `data/train_casos_recuperados.csv` |
| 3 | B1 com e sem esses 50 casos, avaliado nos **mesmos** casos de validação | `results/notebook1b/recuperados_*.csv` |
| 4 | Palavras com maior peso no B1 por especialidade (as pistas que o modelo usa) | `results/notebook1b/b1_top_palavras.csv` |

**B1** = melhor configuração da pessoa 1 (`all_fields_balanced_class_weights`): SVM linear com C=0.5, palavras (uni+bigramas) em todos os campos e fragmentos de 3–5 caracteres nos campos curtos (peso 0.5), pesos de classe q=1 e 1.ª keyword retirada quando é a especialidade.

## 1. Verificação dos dados

In [1]:
import sys, csv, io, json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root / 'src'))
from parte1_base import prepare_data, create_model, class_weights, measure, normalize, specialties, csv_fields

data_folder = root / 'data'
study_folder = root / 'results' / 'notebook1b'
study_folder.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(data_folder / 'train.csv', sep=';', keep_default_na=False)
data = prepare_data(raw)
splits = json.loads((root / 'results' / 'notebook1' / 'splits_seed42.json').read_text(encoding='utf-8'))['folds']
fold_of = {row_id: s['fold'] for s in splits for row_id in s['evaluation']}
data['fold'] = data['row_id'].map(fold_of)

print(f'pandas reads {len(raw)} rows; {len(raw) - len(data)} have a broken label (CSV quote errors) -> {len(data)} cases')
print(f"{data['group_id'].nunique()} groups of identical description/transcription")
pd.DataFrame({'cases': data['medical_specialty'].value_counts(),
              'share': data['medical_specialty'].value_counts(normalize=True).round(3)})

pandas reads 2616 rows; 3 have a broken label (CSV quote errors) -> 2613 cases
1817 groups of identical description/transcription


,cases,share
medical_specialty,,
Surgery,876,0.335
Cardiovascular-Pulmonary,299,0.114
Orthopedic,292,0.112
Radiology,217,0.083
General Medicine,207,0.079
Gastroenterology,192,0.073
Neurology,181,0.069
Obstetrics-Gynecology,135,0.052
Neurosurgery,78,0.030


In [2]:
#Empty fields per column
empty = {field: int(data[field].str.strip().eq('').sum()) for field in csv_fields}
print('empty fields:', empty)

#Repeated texts must stay inside one fold, otherwise validation is inflated
for field in ['description', 'transcription']:
    keys = data[field].map(normalize)
    repeated = data[keys.ne('') & keys.duplicated(keep=False)]
    folds_per_text = repeated.groupby(repeated[field].map(normalize))['fold'].nunique()
    labels_per_text = repeated.groupby(repeated[field].map(normalize))['medical_specialty'].nunique()
    print(f'{field}: {len(repeated)} cases share their text with another case; '
          f'{(folds_per_text > 1).sum()} repeated texts cross folds; '
          f'{(labels_per_text > 1).sum()} repeated texts have more than one label')

empty fields: {'description': 1, 'sample_name': 4, 'keywords': 441, 'transcription': 30}
description: 1504 cases share their text with another case; 0 repeated texts cross folds; 703 repeated texts have more than one label


transcription: 1469 cases share their text with another case; 0 repeated texts cross folds; 701 repeated texts have more than one label


## 2. Os 50 casos recuperados do CSV
Algumas linhas do `train.csv` abrem aspas que não fecham. O pandas continua a ler até encontrar outras aspas e cola as linhas seguintes dentro de um só campo, por isso esses casos desaparecem.

A solução é ler o ficheiro **linha a linha**: cada registo começa pelo nome de uma especialidade seguido de `;`. Os registos lidos assim que o pandas não tem são os casos recuperados. As linhas que continuam partidas não entram.

In [3]:
lines = [l for l in (data_folder / 'train.csv').read_text(encoding='utf-8').splitlines()[1:] if l]
starts_record = lambda line: any(line.startswith(s + ';') for s in specialties)

records, unparsed, i = [], 0, 0
while i < len(lines):
    line, j = lines[i], i + 1
    while j < len(lines) and not starts_record(lines[j]):   #continuation of a record split over two lines
        line, j = line + ' ' + lines[j], j + 1
    fields = next(csv.reader(io.StringIO(line, newline=''), delimiter=';', quotechar='"'))
    if len(fields) == 5:
        records.append(fields)
    else:
        unparsed += 1
    i = j
line_by_line = pd.DataFrame(records, columns=['medical_specialty', 'description', 'sample_name', 'transcription', 'keywords'])

key = lambda d: d['medical_specialty'] + '||' + d['description'].map(normalize) + '||' + d['transcription'].map(normalize)
missing = Counter(key(line_by_line)) - Counter(key(data))
recovered = line_by_line[key(line_by_line).isin(missing)].reset_index(drop=True)
recovered.to_csv(data_folder / 'train_casos_recuperados.csv', sep=';', index=False)

print(f'line by line: {len(line_by_line)} records ({unparsed} broken lines left out)')
print(f'recovered (not in the pandas data): {len(recovered)}')
recovered['medical_specialty'].value_counts()

line by line: 2656 records (13 broken lines left out)
recovered (not in the pandas data): 50


medical_specialty
Surgery                     16
Cardiovascular-Pulmonary    10
Orthopedic                   8
Radiology                    4
Gastroenterology             3
General Medicine             3
Ophthalmology                2
Neurology                    2
Neurosurgery                 1
Obstetrics-Gynecology        1
Name: count, dtype: int64

## 3. Os casos recuperados mudam as métricas?
Há dois treinos do B1 nas mesmas 5 divisões, e **a validação é exatamente a mesma** (os 2613 casos originais):
- **sem:** só os casos originais no treino;
- **com:** os casos originais + os 50 recuperados no treino.

Um caso recuperado só entra no treino de uma divisão se a sua descrição ou transcrição não aparecer na validação dessa divisão. Assim não há fuga.

In [4]:
from threadpoolctl import threadpool_limits

def train_b1(train):
    model = create_model(representation='both')   #person 1's defaults: SVM C=0.5, char weight 0.5
    model.fit(train, train['medical_specialty'], classifier__sample_weight=class_weights(list(train['medical_specialty']), q=1))
    return model

predictions = {'without': np.empty(len(data), dtype=object), 'with': np.empty(len(data), dtype=object)}
added = []
with threadpool_limits(limits=2):
    for split in splits:
        train = data[data['row_id'].isin(split['train'])]
        valid = data[data['row_id'].isin(split['evaluation'])]
        seen = set(valid['description'].map(normalize)) | set(valid['transcription'].map(normalize))
        seen.discard('')
        safe = recovered[~recovered['description'].map(normalize).isin(seen) & ~recovered['transcription'].map(normalize).isin(seen)]
        added.append(len(safe))
        for name, train_set in [('without', train), ('with', pd.concat([train, safe], ignore_index=True))]:
            predictions[name][valid.index] = train_b1(train_set).predict(valid)
print('recovered cases added to training per fold:', added)

gold = data['medical_specialty'].values
comparison = pd.DataFrame({name: {k: round(v, 3) for k, v in measure(gold, pred).items() if k in ['accuracy', 'macro_f1', 'min_f1']}
                           for name, pred in predictions.items()}).T
per_class = pd.DataFrame({name: pd.DataFrame(measure(gold, pred)['per_class']).set_index('specialty')['f1'] for name, pred in predictions.items()}).round(3)
per_class['difference'] = per_class['with'] - per_class['without']
comparison.to_csv(study_folder / 'recuperados_metricas.csv')
per_class.to_csv(study_folder / 'recuperados_f1_por_classe.csv')
print(f"{(predictions['with'] != predictions['without']).sum()} of {len(data)} validation predictions change")
comparison

recovered cases added to training per fold: [47, 44, 45, 41, 41]
62 of 2613 validation predictions change


,accuracy,macro_f1,min_f1
without,0.593,0.631,0.447
with,0.591,0.630,0.453


In [5]:
per_class.sort_values('difference')

,without,with,difference
specialty,,,
Psychiatry-Psychology,0.860,0.851,-0.009
Orthopedic,0.595,0.587,-0.008
Gastroenterology,0.644,0.637,-0.007
Surgery,0.469,0.464,-0.005
Obstetrics-Gynecology,0.643,0.641,-0.002
Neurology,0.586,0.585,-0.001
Radiology,0.522,0.521,-0.001
Dermatology,0.625,0.627,0.002
Cardiovascular-Pulmonary,0.648,0.651,0.003


## 4. Palavras com maior peso no B1
O B1 é treinado com os 2613 casos. Para cada especialidade, o SVM dá um peso a cada palavra ou fragmento: os maiores pesos são as pistas que mais empurram para essa especialidade. O prefixo diz de que campo vem a pista (`description_word`, `keywords_char`, ...).

In [6]:
with threadpool_limits(limits=2):
    b1 = train_b1(data)

tfidf, svm = b1.named_steps['tfidf'], b1.named_steps['classifier']
feature_names = np.concatenate([[f'{name}: {term}' for term in vectorizer.get_feature_names_out()]
                                for name, vectorizer, _ in tfidf.transformers_ if name != 'remainder'])
assert len(feature_names) == svm.coef_.shape[1]

TOP = 15
rows = []
for k, specialty in enumerate(svm.classes_):
    for rank, j in enumerate(np.argsort(svm.coef_[k])[::-1][:TOP], 1):
        block, term = feature_names[j].split(': ', 1)
        rows.append({'specialty': specialty, 'rank': rank, 'block': block, 'term': term, 'weight': round(svm.coef_[k, j], 3)})
top = pd.DataFrame(rows)
top.to_csv(study_folder / 'b1_top_palavras.csv', index=False)

#Word features only (easier to read in the paper): top 8 per specialty
(top[top['block'].str.endswith('_word')].groupby('specialty').head(8)
    .groupby('specialty')['term'].agg(', '.join).to_frame('top words (word blocks)'))

,top words (word blocks)
specialty,
Cardiovascular-Pulmonary,"cardiac, pulmonary, lung, tachypnea, bronchosc..."
Dermatology,"acne, acne, skin biopsy, cream, vulgaris, acne..."
Gastroenterology,"gi, colonoscopy, gi consultation, abdominal, e..."
General Medicine,"gen, gen med, med, med consult, normal, snorin..."
Neurology,"neurologic, head, electroencephalogram, hydroc..."
Neurosurgery,"craniotomy, facetectomy, foraminotomy, hemicra..."
Obstetrics-Gynecology,"ob/gyn, vaginal, hysterectomy, laparoscopy, de..."
Ophthalmology,"eye, eye, eye, blepharoplasty, ophthalmology, ..."
Orthopedic,"pain, hip, orthopedic, back, pain, fracture, k..."


## Notas para o relatório
- **Casos recuperados:** ver secção 3. Se a diferença for pequena, basta uma frase em Data Analysis ("o CSV tem aspas mal fechadas; recuperámos 50 casos e o efeito foi …"). É uma observação concreta sobre os dados, que conta pontos.
- **Pistas do B1:** ver secção 4. São úteis para explicar os exemplos da Discussão (que palavras puxaram o modelo para a etiqueta errada).